# Sign Language Recognition Using Deep Convolutional Neural Networks

SCU9M6 NLP&CV Assignment, Spring 2026.

This notebook implements data loading, preprocessing, a baseline CNN, an enhanced VGG-style CNN, hyperparameter comparison, training curves, confusion matrix, classification report, and sample predictions for the Sign Language MNIST dataset.

## Dataset Setup

Download the Kaggle dataset from https://www.kaggle.com/datasets/datamunge/sign-language-mnist and place these files in the `data/` directory:

- `data/sign_mnist_train.csv`
- `data/sign_mnist_test.csv`

The notebook keeps the official Kaggle test file untouched and creates a stratified validation split from the training file.

In [ ]:
import json
import random
import time
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
import tensorflow as tf
from sklearn.metrics import classification_report, confusion_matrix
from sklearn.model_selection import train_test_split
from tensorflow.keras import layers, regularizers
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau

try:
    from IPython.display import display
except ImportError:
    display = print

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)

sns.set_theme(style='whitegrid')
print('TensorFlow version:', tf.__version__)
print('Available devices:', tf.config.list_physical_devices())

## 1. Data Loading and Preprocessing

In [ ]:
PROJECT_DIR = Path.cwd()
DATA_DIR = PROJECT_DIR / 'data'
FIG_DIR = PROJECT_DIR / 'figures'
OUT_DIR = PROJECT_DIR / 'outputs'
for directory in (DATA_DIR, FIG_DIR, OUT_DIR):
    directory.mkdir(exist_ok=True)

TRAIN_CSV = DATA_DIR / 'sign_mnist_train.csv'
TEST_CSV = DATA_DIR / 'sign_mnist_test.csv'

ORIGINAL_LABELS = [i for i in range(25) if i != 9]
CLASS_NAMES = [chr(ord('A') + i) for i in ORIGINAL_LABELS]
NUM_CLASSES = len(CLASS_NAMES)
LABEL_TO_INDEX = {label: idx for idx, label in enumerate(ORIGINAL_LABELS)}
INDEX_TO_LABEL = {idx: label for label, idx in LABEL_TO_INDEX.items()}

print('Class names:', CLASS_NAMES)
print('Number of classes:', NUM_CLASSES)

In [ ]:
def load_sign_mnist_csv(csv_path):
    if not csv_path.exists():
        raise FileNotFoundError(f'Missing dataset file: {csv_path}. Download the Kaggle CSV and place it in the data directory.')
    df = pd.read_csv(csv_path)
    if 'label' not in df.columns:
        raise ValueError(f'{csv_path.name} must contain a label column.')
    y_original = df['label'].to_numpy(dtype=np.int64)
    unknown_labels = sorted(set(y_original) - set(ORIGINAL_LABELS))
    if unknown_labels:
        raise ValueError(f'Unexpected labels found: {unknown_labels}')
    pixels = df.drop(columns=['label']).to_numpy(dtype=np.float32)
    if pixels.shape[1] != 784:
        raise ValueError(f'Expected 784 pixel columns, found {pixels.shape[1]}.')
    x = pixels.reshape((-1, 28, 28, 1)) / 255.0
    y = np.array([LABEL_TO_INDEX[int(label)] for label in y_original], dtype=np.int64)
    return x, y, y_original, df

X_train_full, y_train_full, y_train_original, train_df = load_sign_mnist_csv(TRAIN_CSV)
X_test, y_test, y_test_original, test_df = load_sign_mnist_csv(TEST_CSV)

X_train, X_val, y_train, y_val = train_test_split(
    X_train_full,
    y_train_full,
    test_size=0.15,
    random_state=SEED,
    stratify=y_train_full,
)

print('Full training set:', X_train_full.shape, y_train_full.shape)
print('Training split:', X_train.shape, y_train.shape)
print('Validation split:', X_val.shape, y_val.shape)
print('Test set:', X_test.shape, y_test.shape)
print('Pixel range after normalisation:', float(X_train.min()), float(X_train.max()))

In [ ]:
def save_class_distribution():
    counts = pd.Series(y_train_full).value_counts().sort_index()
    labels = [CLASS_NAMES[i] for i in counts.index]
    plt.figure(figsize=(11, 4))
    sns.barplot(x=labels, y=counts.values, color='#4c78a8')
    plt.title('Training Class Distribution')
    plt.xlabel('ASL letter')
    plt.ylabel('Number of images')
    plt.tight_layout()
    path = FIG_DIR / 'class_distribution.png'
    plt.savefig(path, dpi=160, bbox_inches='tight')
    plt.show()
    return path

def save_sample_grid():
    selected = []
    for class_idx in range(NUM_CLASSES):
        matches = np.where(y_train_full == class_idx)[0]
        if len(matches) > 0:
            selected.append(matches[0])
    fig, axes = plt.subplots(4, 6, figsize=(9, 6))
    for ax, image_idx in zip(axes.flat, selected):
        ax.imshow(X_train_full[image_idx].squeeze(), cmap='gray')
        ax.set_title(CLASS_NAMES[y_train_full[image_idx]])
        ax.axis('off')
    plt.suptitle('Example Image from Each Class', y=1.02)
    plt.tight_layout()
    path = FIG_DIR / 'sample_images.png'
    plt.savefig(path, dpi=160, bbox_inches='tight')
    plt.show()
    return path

class_distribution_path = save_class_distribution()
sample_grid_path = save_sample_grid()
print('Saved:', class_distribution_path)
print('Saved:', sample_grid_path)

## 2. CNN Architectures

The baseline CNN is intentionally compact. The enhanced CNN follows a VGG-style pattern with repeated 3 by 3 convolutions, batch normalisation, dropout, L2 regularisation, and light augmentation.

In [ ]:
def make_optimizer(name='adam', learning_rate=1e-3):
    name = name.lower()
    if name == 'adam':
        return tf.keras.optimizers.Adam(learning_rate=learning_rate)
    if name == 'sgd':
        return tf.keras.optimizers.SGD(learning_rate=learning_rate, momentum=0.9)
    if name == 'rmsprop':
        return tf.keras.optimizers.RMSprop(learning_rate=learning_rate)
    raise ValueError(f'Unsupported optimizer: {name}')

def compile_model(model, optimizer_name='adam', learning_rate=1e-3):
    model.compile(
        optimizer=make_optimizer(optimizer_name, learning_rate),
        loss='sparse_categorical_crossentropy',
        metrics=['accuracy'],
    )
    return model

def build_baseline_cnn(learning_rate=1e-3, optimizer_name='adam', dropout_rate=0.30):
    inputs = layers.Input(shape=(28, 28, 1))
    x = layers.Conv2D(32, 3, padding='same', activation='relu', kernel_initializer='he_normal')(inputs)
    x = layers.MaxPooling2D(pool_size=2)(x)
    x = layers.Conv2D(64, 3, padding='same', activation='relu', kernel_initializer='he_normal')(x)
    x = layers.MaxPooling2D(pool_size=2)(x)
    x = layers.Flatten()(x)
    x = layers.Dense(128, activation='relu', kernel_initializer='he_normal')(x)
    x = layers.Dropout(dropout_rate)(x)
    outputs = layers.Dense(NUM_CLASSES, activation='softmax')(x)
    model = tf.keras.Model(inputs, outputs, name='baseline_cnn')
    return compile_model(model, optimizer_name, learning_rate)

def conv_bn_relu(x, filters, l2_strength):
    x = layers.Conv2D(
        filters,
        3,
        padding='same',
        activation='relu',
        kernel_initializer='he_normal',
        kernel_regularizer=regularizers.l2(l2_strength),
    )(x)
    return layers.BatchNormalization()(x)

def build_vgg_style_cnn(learning_rate=1e-3, optimizer_name='adam', dropout_rate=0.40, l2_strength=1e-4):
    inputs = layers.Input(shape=(28, 28, 1))
    x = inputs
    for filters in (32, 64, 128):
        x = conv_bn_relu(x, filters, l2_strength)
        x = conv_bn_relu(x, filters, l2_strength)
        x = layers.MaxPooling2D(pool_size=2)(x)
        block_dropout = dropout_rate / 2 if filters < 128 else dropout_rate
        x = layers.Dropout(block_dropout)(x)
    x = layers.GlobalAveragePooling2D()(x)
    x = layers.Dense(
        128,
        activation='relu',
        kernel_initializer='he_normal',
        kernel_regularizer=regularizers.l2(l2_strength),
    )(x)
    x = layers.Dropout(dropout_rate)(x)
    outputs = layers.Dense(NUM_CLASSES, activation='softmax')(x)
    model = tf.keras.Model(inputs, outputs, name='vgg_style_cnn')
    return compile_model(model, optimizer_name, learning_rate)

baseline_preview = build_baseline_cnn()
vgg_preview = build_vgg_style_cnn()
print('Baseline parameters:', baseline_preview.count_params())
print('Enhanced VGG-style parameters:', vgg_preview.count_params())

## 3. Training, Testing, Evaluation, and Hyperparameter Tuning

In [ ]:
EPOCHS = None

EXPERIMENTS = [
    {
        'name': 'baseline_adam_lr1e-3_bs128',
        'builder': build_baseline_cnn,
        'optimizer_name': 'adam',
        'learning_rate': 1e-3,
        'batch_size': 128,
        'dropout_rate': 0.30,
        'epochs': 8,
    },
    {
        'name': 'baseline_adam_lr5e-4_bs256',
        'builder': build_baseline_cnn,
        'optimizer_name': 'adam',
        'learning_rate': 5e-4,
        'batch_size': 256,
        'dropout_rate': 0.30,
        'epochs': 8,
    },
    {
        'name': 'vgg_style_adam_lr1e-3_bs128',
        'builder': build_vgg_style_cnn,
        'optimizer_name': 'adam',
        'learning_rate': 1e-3,
        'batch_size': 128,
        'dropout_rate': 0.40,
        'l2_strength': 1e-4,
        'epochs': 10,
    },
]

def plot_training_curves(history, model_name):
    fig, axes = plt.subplots(1, 2, figsize=(12, 4))
    axes[0].plot(history.history['accuracy'], label='Training accuracy')
    axes[0].plot(history.history['val_accuracy'], label='Validation accuracy')
    axes[0].set_title(f'{model_name}: Accuracy')
    axes[0].set_xlabel('Epoch')
    axes[0].set_ylabel('Accuracy')
    axes[0].legend()
    axes[1].plot(history.history['loss'], label='Training loss')
    axes[1].plot(history.history['val_loss'], label='Validation loss')
    axes[1].set_title(f'{model_name}: Loss')
    axes[1].set_xlabel('Epoch')
    axes[1].set_ylabel('Loss')
    axes[1].legend()
    plt.tight_layout()
    path = FIG_DIR / f'training_curves_{model_name}.png'
    plt.savefig(path, dpi=160, bbox_inches='tight')
    plt.show()
    return path

def train_one_experiment(config):
    tf.keras.backend.clear_session()
    kwargs = {
        'learning_rate': config['learning_rate'],
        'optimizer_name': config['optimizer_name'],
        'dropout_rate': config['dropout_rate'],
    }
    if 'l2_strength' in config:
        kwargs['l2_strength'] = config['l2_strength']
    model = config['builder'](**kwargs)
    callbacks = [
        EarlyStopping(monitor='val_loss', patience=4, restore_best_weights=True),
        ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=2, min_lr=1e-5),
    ]
    start_time = time.time()
    history = model.fit(
        X_train,
        y_train,
        validation_data=(X_val, y_val),
        epochs=config['epochs'],
        batch_size=config['batch_size'],
        callbacks=callbacks,
        verbose=2,
    )
    training_time = time.time() - start_time
    val_loss, val_accuracy = model.evaluate(X_val, y_val, verbose=0)
    test_loss, test_accuracy = model.evaluate(X_test, y_test, verbose=0)
    curve_path = plot_training_curves(history, config['name'])
    model_path = OUT_DIR / f"{config['name']}.keras"
    model.save(model_path)
    row = {
        'model': config['name'],
        'optimizer': config['optimizer_name'],
        'learning_rate': config['learning_rate'],
        'batch_size': config['batch_size'],
        'dropout_rate': config['dropout_rate'],
        'epochs_run': len(history.history['loss']),
        'parameters': model.count_params(),
        'validation_loss': val_loss,
        'validation_accuracy': val_accuracy,
        'test_loss': test_loss,
        'test_accuracy': test_accuracy,
        'training_time_seconds': training_time,
        'curve_path': str(curve_path),
        'model_path': str(model_path),
    }
    return model, history, row

models = {}
histories = {}
results = []

for config in EXPERIMENTS:
    print('\n' + '=' * 80)
    print('Training:', config['name'])
    model, history, row = train_one_experiment(config)
    models[config['name']] = model
    histories[config['name']] = history.history
    results.append(row)

comparison_df = pd.DataFrame(results).sort_values('test_accuracy', ascending=False).reset_index(drop=True)
comparison_path = OUT_DIR / 'model_comparison.csv'
comparison_df.to_csv(comparison_path, index=False)
display(comparison_df)
print('Saved comparison table:', comparison_path)

## Detailed Evaluation of the Best Model

In [ ]:
best_name = comparison_df.loc[0, 'model']
best_model = models[best_name]
print('Best model:', best_name)

y_prob = best_model.predict(X_test, batch_size=128)
y_pred = np.argmax(y_prob, axis=1)

report = classification_report(y_test, y_pred, target_names=CLASS_NAMES, output_dict=True)
report_df = pd.DataFrame(report).transpose()
report_path = OUT_DIR / 'classification_report_best.csv'
report_df.to_csv(report_path)
display(report_df)
print('Saved classification report:', report_path)

cm = confusion_matrix(y_test, y_pred)
plt.figure(figsize=(11, 9))
sns.heatmap(cm, annot=False, cmap='Blues', xticklabels=CLASS_NAMES, yticklabels=CLASS_NAMES)
plt.title(f'Confusion Matrix: {best_name}')
plt.xlabel('Predicted label')
plt.ylabel('True label')
plt.tight_layout()
cm_path = FIG_DIR / 'confusion_matrix_best.png'
plt.savefig(cm_path, dpi=180, bbox_inches='tight')
plt.show()
print('Saved confusion matrix:', cm_path)

In [ ]:
rng = np.random.default_rng(SEED)
sample_indices = rng.choice(len(X_test), size=12, replace=False)

fig, axes = plt.subplots(3, 4, figsize=(10, 8))
for ax, idx in zip(axes.flat, sample_indices):
    true_label = CLASS_NAMES[y_test[idx]]
    predicted_label = CLASS_NAMES[y_pred[idx]]
    color = 'green' if true_label == predicted_label else 'red'
    ax.imshow(X_test[idx].squeeze(), cmap='gray')
    ax.set_title(f'True: {true_label} | Pred: {predicted_label}', color=color, fontsize=10)
    ax.axis('off')
plt.suptitle(f'Sample Predictions: {best_name}', y=1.02)
plt.tight_layout()
pred_path = FIG_DIR / 'sample_predictions_best.png'
plt.savefig(pred_path, dpi=180, bbox_inches='tight')
plt.show()
print('Saved sample predictions:', pred_path)

In [ ]:
summary = {
    'best_model': best_name,
    'comparison_table': str(comparison_path),
    'classification_report': str(report_path),
    'confusion_matrix': str(cm_path),
    'sample_predictions': str(pred_path),
    'class_distribution': str(class_distribution_path),
    'sample_images': str(sample_grid_path),
}
summary_path = OUT_DIR / 'report_artifacts.json'
with open(summary_path, 'w', encoding='utf-8') as f:
    json.dump(summary, f, indent=2)
print(json.dumps(summary, indent=2))

## 4. Reflection Notes

Use the generated comparison table, classification report, confusion matrix, and sample predictions to complete the report. Discuss whether the baseline underfits, whether the enhanced CNN overfits, which letters are confused, and whether the enhanced model's extra training time is justified by improved accuracy or F1-score.